# 📊 Personal Finance Analysis

| Module | Responsibility |
|---|---|
| `categorizer.py` | Reads raw bank file → asks for unknown categories → saves categorised CSV |
| `storage.py` | Appends categorised CSV to the global Excel, deduplicates, loads all data |
| `processor.py` | Aggregates transactions into monthly stats (supports date range filter) |
| `charts.py` | All matplotlib figures |
| `anomaly.py` | Detects anomalous months, redistributes excess, adjusted stats |

**Monthly workflow:**
1. Download bank export → run **Cell 1** (categorise)
2. Review table in **Cell 2**, fix any wrong categories
3. Run **Cell 3** to append to the global Excel
4. Run **Cells 4–7** for analysis

## 0 · Setup

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from style    import apply_dark_theme
from storage  import review, save_to_global, load_from_global, global_summary
from processor import process, print_summary
from charts   import plot_spending_analysis, plot_investment_allocation
from anomaly  import detect, print_report, adjusted_process, plot_anomaly_overview

apply_dark_theme()

## 1 · Categorise new bank export

Run this cell **alone** (Shift+Enter) when you have a new monthly file from your bank.
It auto-fills known merchants and asks about new ones interactively.

In [ ]:
from categorizer import categorize_csv

# --- EDIT THESE ---------------------------------------------------
RAW_FILE   = "mayo.xlsx"               # file from your bank (.xlsx or .csv)
BANK       = "bbva"                    # "bbva" or "myinvestor"
OUT_CSV    = "bbva_mayo_cat.csv"        # temporary categorised output
# ------------------------------------------------------------------

out_path = categorize_csv(RAW_FILE, bank=BANK, output_path=OUT_CSV)

## 2 · Review categorised transactions

Check the table below. You can edit the `Category` column in the DataFrame
before saving, or re-run Cell 1 after updating `comercios_conocidos.json`.

In [ ]:
df_new = review(OUT_CSV)
df_new   # displays the table

## 3 · Append to global Excel

Only run this once you are happy with the categories above.
Duplicate rows (same date + amount + description + source) are silently skipped,
so running this cell twice is safe.

In [ ]:
# --- EDIT: label shown in the Source column of the global file ----
SOURCE = "BBVA"   # or "MyInvestor", "Trade Republic"
# ------------------------------------------------------------------

save_to_global(df_new, source=SOURCE)

# Quick overview of the global file
global_summary()

## 4 · Load all data & configure analysis period

Set `START_MONTH` / `END_MONTH` to focus on a specific period.
Leave as `None` to use all available data.

In [ ]:
all_data = load_from_global()

# --- EDIT: analysis period ----------------------------------------
START_MONTH = None          # e.g. "2025-01"  or  None for all data
END_MONTH   = None          # e.g. "2025-12"  or  None for all data

# Months to always exclude (incomplete data, etc.)
EXCLUDE_MONTHS = {
    "2022-04", "2022-05", "2022-06", "2022-07", "2022-08", "2022-09",
    "2022-10", "2022-11", "2022-12",
    "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06",
    "2024-06",
}
# ------------------------------------------------------------------

d = process(
    all_data,
    exclude_months=EXCLUDE_MONTHS,
    start_month=START_MONTH,
    end_month=END_MONTH,
)
print(f"\n✅ {len(d['all_months_str'])} months in analysis  "
      f"({d['all_months_str'][0]} → {d['all_months_str'][-1]})")

## 5 · Summary statistics

In [ ]:
ACCOUNTS_LABEL = "MyInvestor · Trade Republic · BBVA"
PERIOD_LABEL   = f"{d['all_months_str'][0]} – {d['all_months_str'][-1]}"

print_summary(d, accounts_label=ACCOUNTS_LABEL, period_label=PERIOD_LABEL)

## 6 · Charts

In [ ]:
fig1 = plot_spending_analysis(d, subtitle=f"{ACCOUNTS_LABEL}  ·  {PERIOD_LABEL}")
fig1.show()

In [ ]:
fig2 = plot_investment_allocation(d)
fig2.show()

---
## 7 · (Optional) Anomalous month analysis

Months with unusually high or low spending are flagged using the IQR method.
The excess is redistributed across normal months to give a cleaner budget baseline.

Adjust `IQR_MULTIPLIER`:
- `1.5` — standard (flags moderately high months)
- `3.0` — only extreme outliers

In [ ]:
# --- EDIT ---------------------------------------------------------
IQR_MULTIPLIER = 1.5
# ------------------------------------------------------------------

anomalies = detect(d, iqr_multiplier=IQR_MULTIPLIER)
print_report(d, anomalies)

In [ ]:
# Bar chart with anomalous months highlighted in red
fig3 = plot_anomaly_overview(d, anomalies)
fig3.show()

In [ ]:
# Re-run full analysis EXCLUDING anomalous months
# Averages and std devs will be more representative of your normal spending.

d_clean = adjusted_process(all_data, anomalies, exclude_months=EXCLUDE_MONTHS,
                           start_month=START_MONTH, end_month=END_MONTH)
print_summary(d_clean, accounts_label=ACCOUNTS_LABEL,
              period_label=f"{PERIOD_LABEL}  [anomalies excluded]")